# 第六部分 泰坦尼克号乘客数据分析报告

## 项目概述
- 数据来源：数据来源于机器学习竞赛网站kaggle
- 事件：泰坦尼克号邮轮沉没事件
- 分析目的：探索乘客数据的每个特征的分布以及与是否获救之间的关系

## 数据初探
- 数据集的整体状况
- 是否有缺失值、异常值
- 数据的取值范围
- 每个特征的含义

In [2]:
# 准备工作
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
## 设置中文显示
plt.rcParams["font.sans-serif"] = "SimHei"
## 设置正常显示符号
plt.rcParams["axes.unicode_minus"] = False

In [3]:
# 读入数据
df = pd.read_csv("data/titanic_data.csv")
df.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,1.0,1.0,"Allen, Miss. Elisabeth Walton",female,29.0000,0.0,0.0,24160,211.3375,B5,S
1,1.0,1.0,"Allison, Master. Hudson Trevor",male,0.9167,1.0,2.0,113781,151.5500,C22 C26,S
2,1.0,0.0,"Allison, Miss. Helen Loraine",female,2.0000,1.0,2.0,113781,151.5500,C22 C26,S
3,1.0,0.0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1.0,2.0,113781,151.5500,C22 C26,S
4,1.0,0.0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1.0,2.0,113781,151.5500,C22 C26,S


### 特征含义
- pclass：船舱等级（1、2、3等舱）
- survived：是否获救（1-获救，2-遇难）
- name：乘客姓名（包含姓名中的Title）
- sex：乘客性别
- sibsp：（堂）兄弟姐妹的人数
- parch：直系亲属人数
- ticket：船票编号
- fare：票价
- cabin：船舱编号
- embarked：登船港口（S-南安普顿，C-瑟堡，Q-皇后镇）


In [5]:
df.tail()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
1305,3.0,0.0,"Zabour, Miss. Thamine",female,NaN,1.0,0.0,2665,14.4542,NaN,C
1306,3.0,0.0,"Zakarian, Mr. Mapriededer",male,26.5,0.0,0.0,2656,7.2250,NaN,C
1307,3.0,0.0,"Zakarian, Mr. Ortin",male,27.0,0.0,0.0,2670,7.2250,NaN,C
1308,3.0,0.0,"Zimmerman, Mr. Leo",male,29.0,0.0,0.0,315082,7.8750,NaN,S
1309,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# 删除空行
df.drop(1309,axis=0,inplace=True)

In [8]:
# 数据基本信息
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1309 entries, 0 to 1308
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    1309 non-null   float64
 1   survived  1309 non-null   float64
 2   name      1309 non-null   object 
 3   sex       1309 non-null   object 
 4   age       1046 non-null   float64
 5   sibsp     1309 non-null   float64
 6   parch     1309 non-null   float64
 7   ticket    1309 non-null   object 
 8   fare      1308 non-null   float64
 9   cabin     295 non-null    object 
 10  embarked  1307 non-null   object 
dtypes: float64(6), object(5)
memory usage: 122.7+ KB


### 数据信息
- 乘客总数：1309
- 特征总数：11
- 缺失值：年龄、票价、船舱编号、登船港口

In [9]:
# 数值数据
df.describe()

,pclass,survived,age,sibsp,parch,fare
count,1309.000000,1309.000000,1046.000000,1309.000000,1309.000000,1308.000000
mean,2.294882,0.381971,29.881135,0.498854,0.385027,33.295479
std,0.837836,0.486055,14.413500,1.041658,0.865560,51.758668
min,1.000000,0.000000,0.166700,0.000000,0.000000,0.000000
25%,2.000000,0.000000,21.000000,0.000000,0.000000,7.895800
50%,3.000000,0.000000,28.000000,0.000000,0.000000,14.454200
75%,3.000000,1.000000,39.000000,1.000000,0.000000,31.275000
max,3.000000,1.000000,80.000000,8.000000,9.000000,512.329200


### 数值特征的统计信息
- 平均年龄29.9岁，最大80岁，最小两个月
- 获救比例38.2%
- 一二等舱乘客较少，三等舱超过半数以上
- 大多数人独自一人，没有亲属同行

## 数据预处理
- 包括缺失值数据和数据转换

### 缺失值处理
- 票价：只有一个缺失值，采用前向填充

In [12]:
# 票价缺失值前向填充
df["fare"].fillna(method="ffill",inplace=True)

- 年龄：采用均值填充

In [13]:
# 年龄缺失值
aver_age = np.round(df["age"].mean(),1)
aver_age
df["age"].fillna(aver_age,inplace=True)

- 登船港口：有两个缺失值，采用最多值填充

In [16]:
# 登船港口采用最多值填充
df["embarked"].value_counts()
df["embarked"].fillna("S",inplace=True)

- 船舱编号：缺失值太多，可以删除
- 这里采用参加一列来标注有无船舱编号

In [19]:
# 船舱编号的缺失值：转换为有无编号
df["have_cabin"] = df["cabin"].notnull()
df["have_cabin"].value_counts()

False    1014
True      295
Name: have_cabin, dtype: int64

In [20]:
# 删除船舱编号列
df.drop("cabin",axis=1,inplace=True)

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1309 entries, 0 to 1308
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   pclass      1309 non-null   float64
 1   survived    1309 non-null   float64
 2   name        1309 non-null   object 
 3   sex         1309 non-null   object 
 4   age         1309 non-null   float64
 5   sibsp       1309 non-null   float64
 6   parch       1309 non-null   float64
 7   ticket      1309 non-null   object 
 8   fare        1309 non-null   float64
 9   embarked    1309 non-null   object 
 10  hane_cabin  1309 non-null   bool   
 11  have_cabin  1309 non-null   bool   
dtypes: bool(2), float64(6), object(4)
memory usage: 147.3+ KB
